# Starter notebook

Works in both modes:
- **local** — runs on your machine, MLflow on localhost, DVC pulls from S3
- **cloud** — run this notebook on the EC2 VM or package it as a training script

MLflow and DVC are pre-configured by devenv — no setup needed.

In [1]:
import os
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Both of these are set by devenv — no need to configure manually
print("MLflow tracking URI:", os.environ.get('MLFLOW_TRACKING_URI'))
print("DVC remote URL:     ", os.environ.get('DVC_REMOTE_URL'))
print("Infra mode:         ", os.environ.get('INFRA_MODE', 'local'))

/home/ml/project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow tracking URI: http://localhost:5000
DVC remote URL:      s3://nix-ml-solo-dev-dvc/dvc
Infra mode:          cloud


## 1. Load data

DVC tracks your data. After `dvc pull` your data is in `data/`.

Replace the sample data below with your own dataset.

In [2]:
# Replace with: pd.read_csv('data/train.csv')
from sklearn.datasets import load_iris
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
X.head()

Train: 120 rows  |  Test: 30 rows


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 2. Train with MLflow tracking

In [3]:
# Hyperparameters — tweak these
PARAMS = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
}

mlflow.set_experiment("starter")

with mlflow.start_run() as run:
    print(f"Run ID: {run.info.run_id}")

    mlflow.log_params(PARAMS)

    model = RandomForestClassifier(**PARAMS)
    model.fit(X_train, y_train)

    # ── Evaluation metrics ──────────────────────────────────────────────
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", acc)

    # Per-class precision / recall from classification_report
    from sklearn.metrics import classification_report
    report = classification_report(y_test, y_pred,
                                   target_names=iris.target_names,
                                   output_dict=True)
    for cls, scores in report.items():
        if isinstance(scores, dict):
            for metric, value in scores.items():
                mlflow.log_metric(f"{cls}_{metric}".replace(' ', '_'), value)

    # ── Log inference predictions as a table ────────────────────────────
    predictions_df = X_test.copy()
    predictions_df["actual"]    = [iris.target_names[t] for t in y_test]
    predictions_df["predicted"] = [iris.target_names[p] for p in y_pred]
    for i, cls in enumerate(iris.target_names):
        predictions_df[f"proba_{cls}"] = y_proba[:, i].round(4)
    predictions_df["correct"] = predictions_df["actual"] == predictions_df["predicted"]
    mlflow.log_table(predictions_df, artifact_file="inference/test_predictions.json")

    # ── Log feature importances ─────────────────────────────────────────
    fi_df = pd.DataFrame({
        "feature":   X.columns,
        "importance": model.feature_importances_.round(4),
    }).sort_values("importance", ascending=False)
    mlflow.log_table(fi_df, artifact_file="inference/feature_importances.json")

    # ── Log model ───────────────────────────────────────────────────────
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="starter-model",
    )

    print(f"Accuracy: {acc:.4f}")
    print(f"\nTo deploy this run:")
    print(f"  deploy {run.info.run_id}")

RUN_ID = run.info.run_id


2026/06/25 14:17:02 INFO mlflow.tracking.fluent: Experiment with name 'starter' does not exist. Creating a new experiment.


Run ID: f6a397b4eb944c59bffc16d4be1a0119


2026/06/25 14:17:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/25 14:17:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'starter-model'.
2026/06/25 14:17:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: starter-model, version 1


Accuracy: 1.0000

To deploy this run:
  deploy f6a397b4eb944c59bffc16d4be1a0119
🏃 View run unruly-crane-914 at: http://localhost:5000/#/experiments/1/runs/f6a397b4eb944c59bffc16d4be1a0119
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '1' of model 'starter-model'.


In [4]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## 3. Deploy

From the terminal:

```sh
# local mode — serves on localhost:5001
deploy <run-id>

# cloud mode — packages model + src/inference.py → SageMaker endpoint
deploy <run-id>
```

Edit `src/inference.py` to match your model flavour and set in `devenv.nix`:
```nix
env.INFERENCE_SCRIPT = "src/inference.py";
```

In [5]:
# Quick local test — call the model directly before deploying
sample = X_test.iloc[:3]
predictions = model.predict(sample)
print("Sample predictions:", [iris.target_names[p] for p in predictions])

Sample predictions: [np.str_('versicolor'), np.str_('setosa'), np.str_('virginica')]


## 4. Save data with DVC

After acquiring/transforming data, push it to S3 so the team and SageMaker jobs can access it.

In [6]:
import pathlib

# Save training data
pathlib.Path('data').mkdir(exist_ok=True)
X_train.assign(target=y_train.values).to_csv('data/train.csv', index=False)
X_test.assign(target=y_test.values).to_csv('data/test.csv', index=False)
print('Saved data/train.csv and data/test.csv')

# Version and push to S3 via DVC
import subprocess
subprocess.run(['dvc', 'add', 'data/train.csv', 'data/test.csv'], check=True)
subprocess.run(['dvc', 'push'], check=True)
print('Pushed data to S3')


Saved data/train.csv and data/test.csv


⠋ Checking graph



To track the changes with git, run:

	git add data/test.csv.dvc data/train.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
2 files pushed
Pushed data to S3
